In [1]:
import os
os.chdir("..")

In [ ]:
import polars as pl
import pandas as pd


tu = TradeUtils()

ModuleNotFoundError: No module named 'stem'

In [ ]:
df = pl.read_parquet("data/raw/jp_data.parquet")
df = df.select(cmdcode=pl.col("Commodity_Code").cast(pl.String).str.slice(0,2))
df = df.unique(subset=["cmdcode"]).sort(by="cmdcode")
df = df.slice(1)
df = df.slice(1)
codes = df.to_series().to_list() 
codes

In [ ]:
req_imports = CensusAPI().timeseries_query(
                dataset="timeseries-intltrade-imports-statehs",
                params_list=[
                    "CTY_CODE",
                    "CTY_NAME",
                    "GEN_VAL_MO",
                    "COMM_LVL",
                    "I_COMMODITY",
                ],
                year=_year,
                extra=f"STATE={state}",
                skip_checks=True,
            )
req_imports

In [ ]:
import comtradeapicall

mydf = comtradeapicall._previewFinalData(typeCode='C', freqCode='M', clCode='HS', period='202601',
                                        reporterCode='36', cmdCode='91', flowCode='M', partnerCode="",
                                        partner2Code=None,
                                        customsCode=None, motCode=None, maxRecords=500, format_output='JSON',
                                        aggregateBy=None, breakdownMode='classic', countOnly=None, includeDesc=True)

mydf

In [ ]:
import time
import requests
import polars as pl
from stem import Signal
from stem.control import Controller

# =========================
# TOR CONFIG
# =========================
TOR_SOCKS_PROXY = "socks5h://127.0.0.1:9050"
TOR_CONTROL_PORT = 9051
TOR_PASSWORD = "password"

proxies = {
    "http": TOR_SOCKS_PROXY,
    "https": TOR_SOCKS_PROXY
}

# =========================
# TOR IP RENEWAL
# =========================
def renew_tor_ip():
    try:
        with Controller.from_port(port=TOR_CONTROL_PORT) as controller:
            controller.authenticate(password=TOR_PASSWORD)
            controller.signal(Signal.NEWNYM)
        print("🔄 Tor IP renewed")
        time.sleep(10)  # allow circuit to stabilize
    except Exception as e:
        print(f"❌ Tor renewal failed: {e}")
        time.sleep(15)

# =========================
# TEST TOR CONNECTION
# =========================
def test_tor():
    try:
        r = requests.get("http://httpbin.org/ip", proxies=proxies, timeout=10)
        print("🌍 Current Tor IP:", r.text)
    except Exception as e:
        print("❌ Tor test failed:", e)

# =========================
# SAFE API CALL WRAPPER
# =========================
def fetch_data(year, month, chunk):
    MAX_RETRIES = 5
    retries = 0

    while True:
        try:
            df = comtradeapicall.previewFinalData(
                    typeCode='C',
                    freqCode='M',
                    clCode='HS',
                    period=f'{year}{str(month).zfill(2)}',
                    reporterCode='',
                    cmdCode=','.join(chunk),
                    flowCode='X',
                    partnerCode='584',
                    partner2Code=None,
                    customsCode=None,
                    motCode=None,
                    maxRecords=500,
                    format_output='JSON',
                    aggregateBy=None,
                    breakdownMode=None,
                    countOnly=None,
                    includeDesc=True,
                    proxy_url="http://127.0.0.1:8118"
            )

            if df is None:
                raise ValueError("Empty response")

            return df

        except Exception as e:
            retries += 1
            print(f"⚠️ Error ({retries}): {e}")

            err = str(e).lower()

            if "429" in err or "rate" in err:
                print("🚫 Rate limit → rotating Tor")
                renew_tor_ip()

            elif "proxy" in err or "socks" in err or "connection" in err:
                print("🔌 Connection issue → waiting")
                time.sleep(10)

            else:
                print("❓ Unknown error → rotating Tor")
                renew_tor_ip()

            if retries >= MAX_RETRIES:
                print("❌ Max retries reached, skipping chunk")
                return None

            time.sleep(5)

# =========================
# INITIAL SETUP
# =========================
chunks = [codes[i:i + 50] for i in range(0, len(codes), 50)]

empty_df = [
    pl.Series("refYear", [], dtype=pl.String),
    pl.Series("refMonth", [], dtype=pl.String),
    pl.Series("reporterCode", [], dtype=pl.String),
    pl.Series("reporterDesc", [], dtype=pl.String),
    pl.Series("flowCode", [], dtype=pl.String),
    pl.Series("flowDesc", [], dtype=pl.String),
    pl.Series("partnerDesc", [], dtype=pl.String),
    pl.Series("classificationCode", [], dtype=pl.String),
    pl.Series("cmdCode", [], dtype=pl.String),
    pl.Series("cmdDesc", [], dtype=pl.String),
    pl.Series("cifvalue", [], dtype=pl.String),
    pl.Series("fobvalue", [], dtype=pl.String),
    pl.Series("primaryValue", [], dtype=pl.String),
    pl.Series("netWgt", [], dtype=pl.String),
]

master_df = pl.DataFrame(empty_df)

# =========================
# START
# =========================
test_tor()

for year in range(2010, 2025):
    for month in range(1, 13):

        if year == 2024 and month >= 10:
            continue

        chunk_queue = chunks.copy()

        while chunk_queue:
            chunk = chunk_queue.pop(0)

            df = fetch_data(year, month, chunk)

            if df is None:
                continue

            if df.empty:
                print(f"📭 No data {year}-{month}, chunk {chunk}")
                continue

            # 🚨 HANDLE 500 ROW LIMIT PROPERLY
            if len(df) == 500 and len(chunk) > 1:
                print(f"⚠️ 500 row cap hit → splitting chunk ({len(chunk)})")

                mid = len(chunk) // 2
                chunk_queue.append(chunk[:mid])
                chunk_queue.append(chunk[mid:])
                continue

            # ✅ NORMAL PROCESSING
            df = df[[
                "refYear", "refMonth", "reporterCode", "reporterDesc",
                "flowCode", "flowDesc", "partnerDesc",
                "classificationCode", "cmdCode", "cmdDesc",
                "cifvalue", "fobvalue", "primaryValue", "netWgt"
            ]]

            tmp = pl.from_pandas(df).cast(pl.String)
            master_df = pl.concat([master_df, tmp], how="vertical")

            print(f"✅ Processed {year}-{str(month).zfill(2)} | {len(tmp)} rows")

            # 🧘 Throttle (critical for Tor stability)
            time.sleep(1.5)

# =========================
# SAVE OUTPUT
# =========================
master_df.write_csv("data/master_df.csv")

print("🎉 Done!")